In [1]:
# byte track
import cv2
from ultralytics import YOLO

In [2]:
model = YOLO("yolo26n.pt")
cap = cv2.VideoCapture("road.mp4")

In [3]:
if not cap.isOpened():
    raise RuntimeError("Cannot access the data/ video")

while True:
    ok, frame = cap.read()
    if not ok:
        break
    result = model.track(frame, persist = True, 
                         tracker = "bytetrack.yaml", 
                         conf = 0.1, verbose = False)[0]
    cv2.imshow("YOLO tracking", result.plot())
    if cv2.waitKey(1) & 0xFF==ord("q"):
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
#"Bot - Sort"

In [6]:
model = YOLO("yolo26n.pt")
cap = cv2.VideoCapture("Moving_camera_road.mp4")
if not cap.isOpened():
    raise RuntimeError("Cannot access the data/ video")

while True:
    ok, frame = cap.read()
    if not ok:
        break
    result = model.track(frame, persist = True, 
                         tracker = "botsort.yaml", 
                         conf = 0.1, verbose = False)[0]
    if result.boxes is not None and result.boxes.is_track:
        ids = result.boxes.id.int().cpu().tolist()
        classes = result.boxes.cls.int().cpu().tolist()
        for track_id, cls in zip(ids, classes):
            print("ID:", track_id, "Class: ", result.names[int(cls)])
    output = result.plot()
    output = cv2.resize(output, (640,360))
    cv2.imshow("YOLO -Bot-SORT tracking", output)
    if cv2.waitKey(1) & 0xFF==ord("q"):
        break
cap.release()
cv2.destroyAllWindows()

ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 4 Class:  car
ID: 5 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 19 Class:  car
ID: 4 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 19 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 19 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 19 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 19 Class:  car
ID: 1 Class:  car
ID: 2 Class:  car
ID: 3 Class:  car
ID: 5 Class:  car
ID: 1

In [5]:
# CSRT - Single Object Tracking
# no use of YOLO here

import cv2
import time
video_path = "rush.mp4"
output_path = "CSRT_object_tracking.mp4"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: on opening the file")
    exit()
ret, frame = cap.read()
if not ret:
    print("Could not read the file")
    exit()

# select the vehicle manually
print("Select a vehicle using mouse.")
print("Press enter or Space after selecting.")

bbox = cv2.selectROI("select vehicle", frame, fromCenter=False, showCrosshair = True)

cv2.destroyWindow("select vehicle")

# Create CSRT Tracker
if hasattr(cv2, "TrackerCSRT_create"):
    tracker = cv2.TrackerCSRT_create()
elif hasattr(cv2, "legacy"):
    tracker = cv2.legacy.TrackerCSRT_create()
else:
    raise RuntimeError("CSRT is not available.")
# initalise tracker
tracker.init(frame, bbox)

# video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <=0:
    fps=30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
lost_start_time = None
MAX_LOST_TIME = 1.0 # seconds
while True:
    ret, frame = cap.read()
    if not ret:
         break

    # update CSRT tracker
    success, bbox = tracker.update(frame)

    if success:
        x,y,w,h = [int(v) for v in bbox]
        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0),2)
        cv2.putText(frame, "CSRT Tracking",(x, max(y-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, .7,(0,255,0), 2)
    else:
        #cv2.putText(frame, "Tracking Lost",(x, max(y-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, .7,(0,255,0), 2)
        # ----------------------------------------------------
        # TRACKING LOST
        # ----------------------------------------------------

        if lost_start_time is None:
            lost_start_time = time.time()
        lost_time = time.time() - lost_start_time
        remaining = MAX_LOST_TIME - lost_time
        cv2.putText(frame,"TRACKING LOST",(20, 40),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0, 0, 255),2)
        cv2.putText(frame,f"Quitting in: {max(0, remaining):.1f} sec",(20, 75),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0, 0, 255),2)

        # Quit after 5 seconds
        if lost_time >= MAX_LOST_TIME:
            print("Tracking lost for 1 seconds.")
            print("Quitting...")
            break
    cv2.imshow("CSRT - Single Object Tracking",frame)
    out.write(frame)
    if cv2.waitKey(1) & 0xFF==ord("q"):
        break
cap.release()
out.release()
cv2.destroyAllWindows()    


Select a vehicle using mouse.
Press enter or Space after selecting.
Tracking lost for 1 seconds.
Quitting...


In [5]:
# centroid tracking
video_path = "rush.mp4"
output_path = "Centroid_Object_tracking.mp4"
# COCO class
# car = 2
# motorcycle=3
# bus  = 5
# truck = 7

vehicle_classes = [2,3,5,7]
#confidence = 0.4
model = YOLO("yolo26n.pt") 
cap = cv2.VideoCapture(video_path)
# video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <=0:
    fps=30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
# centroid
centroid = []

while True:
    ok, frame = cap.read()
    if not ok:
        break
    result = model.predict(frame,conf =0.4,classes =vehicle_classes, verbose = False)[0]

    # reset the current frame
    centroid = []
    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        confidence = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        for box,confidence, class_id in zip( boxes, confidence, classes):
            x1,y1,x2,y2 = map(int,box)
            # cal centroid
            cx = int((x1+x2)/2)
            cy = int((y1+y2)/2)

            centroid.append((cx,cy))
            cv2.rectangle(frame, (x1,y1), (x2, y2), (0,255,0),2)
            cv2.circle(frame, (cx,cy), 5, (0,0,255),-1)
            # display centroid
            cv2.putText(frame,f"Centroid: ({cx},{cy})",(x1, max(y1-10,20)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0, 255, 255),2)
        cv2.putText(frame,f"Vechicles: {len(centroid)}",(20, 40),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255, 255),2)
    cv2.imshow("YOLO -Bot-SORT tracking", frame)
    out.write(frame)
    if cv2.waitKey(1) & 0xFF==ord("q"):
        break
cap.release()
out.release()
cv2.destroyAllWindows()